# 00. Frozen Sample, Discretization Rule, and Label Tables (Appendix E.1.1; Table A1; Appendix C.2)

> Reproduction notebook for *Addressing the Psychological Small-Sample Problem with Multimodal Large Language Models: Evidence from Multiple Moral Emotion Recognition Study*. Section numbers refer to the manuscript appendices. Every notebook reads only the committed files in `../final/` and ends with a check against the frozen outputs in `../final/reproducibility/`. Run the notebooks in numerical order; `00` writes the label tables that the others reuse.

This notebook fixes the analysis sample and derives the four-class label of every participant in every configuration. Everything downstream (Appendix E.1–E.6, Appendix D, Tables A1–A3, D1–D2 and E1–E7) starts from the table written here.

**Sample rule (Table A1 note).** A participant enters a configuration when the committed CSV holds a model score on both dimensions and a design label. Nothing is imputed. The resulting valid *N* is the denominator of every accuracy (96–103 depending on the configuration).

**Discretization rule (Appendix E.1.1.1).** A score above the sample mean of its dimension is *high*, a score at or below it is *low*; crossing the two decisions gives Elevation (high/high), Sympathy (high/low), Inspiration (low/high) and Control (low/low). The same rule is applied to every configuration, including the fine-tuned model (Exp 8), the machine-learning fusion output, the observer composites and the still-frame models.

**File-name correspondence.** The CSV files keep the numbering in force when the experiments were run: manuscript Exp 16 (vision + audio + physiology, no speech semantics) is `experiment_14.csv`; the human observer passes are manuscript Exp 14–15 and come from the cleaned rating panel (they were rated outside the experiment notebooks). Class code `INSP` (Inspiration) replaces the earlier `ADMR` label.

In [1]:
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath("."))
import common as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)
rng = np.random.default_rng(C.SEED)
print("final data folder:", os.path.relpath(C.FINAL))

final data folder: ../final


## 1. MLLM configurations (Exp 1–9 and 16)

Rows without a score or a design label are listed as exclusions; the first row is kept if a participant appears twice.

In [2]:
rows = []; excl = []
for e in C.MLLM_EXPS:
    raw = pd.read_csv(os.path.join(C.FINAL, C.MLLM_FILE[e]), low_memory=False)
    d = C.load_mllm_experiment(e)
    pid = pd.to_numeric(raw["Participant_SN_int"], errors="coerce")
    miss_score = raw["sympathy"].isna() | raw["inspiration"].isna(); miss_group = raw["Group"].isna()
    for p in pid[(miss_score | miss_group) & pid.notna()].astype(int): excl.append(dict(exp=e, participant=p, reason="missing score/group"))
    rows.append(dict(exp=e, file=C.MLLM_FILE[e], rows=len(raw), missing_score=int(miss_score.sum()), missing_group=int(miss_group.sum()),
                     valid=len(d), accuracy=round((d.true == d.pred).mean(), 3)))
SAMPLE = pd.DataFrame(rows); SAMPLE

Out[2]: 
   exp               file  rows  missing_score  missing_group  valid  accuracy
0    1   experiment_1.csv   102              4              2     96     0.333
1    2   experiment_2.csv   102              0              2    100     0.490
2    3   experiment_3.csv   102              0              2    100     0.630
3    4   experiment_4.csv   102              0              2    100     0.690
4    5   experiment_5.csv   102              0              2    100     0.660
5    6   experiment_6.csv   102              0              2    100     0.700
6    7   experiment_7.csv   102              0              2    100     0.660
7    8   experiment_8.csv   102              0              2    100     0.660
8    9   experiment_9.csv   102              1              2     99     0.758
9   16  experiment_14.csv   102              0              2    100     0.490


## 2. Machine-learning configurations (Exp 10–13; Appendix C.2)

`machine_learning.csv` holds one Random Forest prediction per participant and modality. Acoustics appears twice (two runs with identical folds; only the correlation-pruning flag differs); the run **with** pruning is kept so that acoustics contributes a single vote. Late fusion takes the mode of the available votes (ties resolved to the smallest class code). The result is compared with the released `experiment_1x_5fold.csv` files.

In [3]:
ml = C.load_ml_predictions()
print("votes per modality:", ml.modality.value_counts().to_dict())
for e in C.ML_EXPS:
    fu = C.fuse_ml(ml, e); rel = pd.read_csv(os.path.join(C.FINAL, f"experiment_{e}_5fold.csv"))
    m = fu.merge(rel, left_on="pid", right_on="user_id")
    agree = int((m.pred == m.predicted_fused.map(lambda v: C.GROUP_NAMES[int(v) + 1])).sum())
    print(f"Exp {e}: N = {len(fu)}, accuracy = {(fu.true == fu.pred).mean():.3f}, agreement with released fusion file: {agree}/{len(m)}")
    rows.append(dict(exp=e, file="machine_learning.csv", rows=len(ml[ml.modality.isin(C.ML_MODALITIES[e])]), missing_score=0, missing_group=0, valid=len(fu), accuracy=round((fu.true == fu.pred).mean(), 3)))
SAMPLE = pd.DataFrame(rows)

votes per modality: {'Speech Text': 101, 'Accoustics': 101, 'Eye_Movement': 101, 'pose': 99, 'Facial': 99, 'r-PPG': 93, 'thermal': 90}
Exp 10: N = 101, accuracy = 0.297, agreement with released fusion file: 101/101
Exp 11: N = 103, accuracy = 0.359, agreement with released fusion file: 103/103
Exp 12: N = 103, accuracy = 0.350, agreement with released fusion file: 103/103
Exp 13: N = 103, accuracy = 0.398, agreement with released fusion file: 103/103


## 3. Human observer passes (Exp 14–15; Appendix C.3 and E.4.6)

The cleaned observer panel (`observer_composite_scores_clean.csv`) gives, for each pass, the three raters' composite scores and their complete-case consensus. Participants with a consensus on both constructs are classified with the same sample-mean rule.

In [4]:
LAB = C.load_all_labels()
for e, ver in [(14, "silent"), (15, "audiovisual")]:
    d = LAB[e]; print(f"Exp {e} ({ver}): N = {len(d)}, consensus accuracy = {(d.true == d.pred).mean():.3f}")
    rows.append(dict(exp=e, file="observer_composite_scores_clean.csv", rows=len(d), missing_score=0, missing_group=0, valid=len(d), accuracy=round((d.true == d.pred).mean(), 3)))
SAMPLE = pd.DataFrame(rows).sort_values("exp").reset_index(drop=True); SAMPLE

Exp 14 (silent): N = 100, consensus accuracy = 0.310
Exp 15 (audiovisual): N = 100, consensus accuracy = 0.730
Out[4]: 
    exp                                 file  rows  missing_score  missing_group  valid  accuracy
0     1                     experiment_1.csv   102              4              2     96     0.333
1     2                     experiment_2.csv   102              0              2    100     0.490
2     3                     experiment_3.csv   102              0              2    100     0.630
3     4                     experiment_4.csv   102              0              2    100     0.690
4     5                     experiment_5.csv   102              0              2    100     0.660
5     6                     experiment_6.csv   102              0              2    100     0.700
6     7                     experiment_7.csv   102              0              2    100     0.660
7     8                     experiment_8.csv   102              0              2    100     0.66

## 4. Write the label tables and verify against the frozen snapshot

In [5]:
ALL = pd.concat([LAB[e].assign(exp=e) for e in sorted(LAB)], ignore_index=True)
ALL.to_csv(os.path.join(C.OUT, "labels_all.csv"), index=False)
SAMPLE.to_csv(os.path.join(C.OUT, "sample_summary.csv"), index=False)
pd.DataFrame(excl).to_csv(os.path.join(C.OUT, "exclusion_list.csv"), index=False)
frozen = C.load_frozen("labels_all_frozen.csv").replace({"ADMR": "INSP"})
m = ALL.merge(frozen, on=["exp", "pid"], suffixes=("", "_frozen"))
print("participant x configuration rows computed:", len(ALL[ALL.exp.isin(frozen.exp.unique())]), "| frozen:", len(frozen), "| matched:", len(m))
print("label disagreements (true):", int((m.true != m.true_frozen).sum()), "| (pred):", int((m.pred != m.pred_frozen).sum()))
snap = C.load_frozen("frozen_sample_snapshot.csv")
chk = SAMPLE.merge(snap[["exp", "valid", "acc_from_csv"]], on="exp", suffixes=("", "_frozen"))
print("valid-N agreement:", int((chk.valid == chk.valid_frozen).sum()), "/", len(chk), "| accuracy max |diff|:", float((chk.accuracy - chk.acc_from_csv).abs().max()))

participant x configuration rows computed: 1405 | frozen: 1405 | matched: 1405
label disagreements (true): 0 | (pred): 0
valid-N agreement: 14 / 14 | accuracy max |diff|: 0.0
